In [1]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 500) 

In [2]:
import onnxruntime as ort
print("onnxruntime providers:", ort.get_available_providers())

import torch
print("torch cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
    print("cuda version:", torch.version.cuda)

onnxruntime providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
torch cuda available: True
cuda device count: 1
cuda device: NVIDIA GeForce RTX 3050 6GB Laptop GPU
cuda version: 12.8


In [3]:
import sys
import os

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_path)

from app.settings import settings
from app.models.sherpa_speaker_diarization import SherpaOnnxSpeakerDiarizationModel

TEST_AUDIO_FILE = settings.AUDIO_DIR / "test_4.mp3"
CACHE_DIR = settings.CACHE_DIR

print("project_path:", project_path)
print("TEST_AUDIO_FILE:", TEST_AUDIO_FILE)
print("CACHE_DIR:", CACHE_DIR)
print("audio exists:", TEST_AUDIO_FILE.exists())

project_path: d:\Projects\speech_to_text_service
TEST_AUDIO_FILE: D:\Projects\speech_to_text_service\data\audio\test_4.mp3
CACHE_DIR: D:\Projects\speech_to_text_service\data\cache_dir
audio exists: True


In [4]:
from pydub import AudioSegment

def get_audio_duration(path: str) -> float:
    audio = AudioSegment.from_file(path)
    return len(audio) / 1000.0

print(f"Audio duration: {get_audio_duration(str(TEST_AUDIO_FILE))} seconds")

Audio duration: 119.14 seconds


In [5]:
progress_values = []

def on_progress(progress: float):
    progress_values.append(progress)
    print(f"\rDiarization progress: {progress:.2f}%", end="")

In [14]:
diarizer = SherpaOnnxSpeakerDiarizationModel(
    cache_dir=CACHE_DIR,
    token=settings.HF_TOKEN,
    provider="cuda",
    cluster_threshold=0.8,
    # min_duration_on=0.3,
    # merge_gap=0.5,
    # min_segment_duration=0.4,
)

2026-07-14 16:00:35 - diarization.sherpa_onnx - INFO - MainProcess[10144] - Модели Sherpa-ONNX diarization готовы | segmentation=D:\Projects\speech_to_text_service\data\cache_dir\models--csukuangfj--sherpa-onnx-pyannote-segmentation-3-0\snapshots\9403a6902bb58e3d5ae8c7e77c3422de279db2e0\model.onnx | embedding=D:\Projects\speech_to_text_service\data\cache_dir\models--csukuangfj--speaker-embedding-models\snapshots\0743f301363dec56491a490f6d6cbc9d67f9a3bf\nemo_en_titanet_small.onnx


In [15]:
result = diarizer.diarize(
    audio_path=TEST_AUDIO_FILE,
    num_speakers=None,
    on_progress=on_progress,
)

print("\nDone")
result

2026-07-14 16:00:35 - diarization.sherpa_onnx - INFO - MainProcess[10144] - Sherpa-ONNX diarizer инициализирован | num_clusters=-1 | sample_rate=16000 | provider=cuda | num_threads=1
2026-07-14 16:00:36 - diarization.sherpa_onnx - INFO - MainProcess[10144] - Sherpa-ONNX diarization запущен | файл=D:\Projects\speech_to_text_service\data\audio\test_4.mp3 | num_speakers=None | sample_rate=16000 | provider=cuda
Diarization progress: 100.00%2026-07-14 16:00:58 - diarization.sherpa_onnx - INFO - MainProcess[10144] - Sherpa-ONNX diarization завершен | файл=D:\Projects\speech_to_text_service\data\audio\test_4.mp3 | спикеров=2 | сегментов=26 | длительность=118.65s | обработка=22.87s

Done


{'duration': 118.64534759521484,
 'num_speakers': 2,
 'speakers': ['SPEAKER_01', 'SPEAKER_02'],
 'segments': [{'start': 0.9590937495231628,
   'end': 12.737844467163086,
   'speaker_id': 1,
   'speaker': 'SPEAKER_01',
   'duration': 11.778750717639923},
  {'start': 13.328469276428223,
   'end': 20.635345458984375,
   'speaker_id': 1,
   'speaker': 'SPEAKER_01',
   'duration': 7.306876182556152},
  {'start': 22.03597068786621,
   'end': 22.93034553527832,
   'speaker_id': 2,
   'speaker': 'SPEAKER_02',
   'duration': 0.8943748474121094},
  {'start': 23.45347023010254,
   'end': 25.259096145629883,
   'speaker_id': 2,
   'speaker': 'SPEAKER_02',
   'duration': 1.8056259155273438},
  {'start': 26.220970153808594,
   'end': 31.401596069335938,
   'speaker_id': 2,
   'speaker': 'SPEAKER_02',
   'duration': 5.180625915527344},
  {'start': 32.025970458984375,
   'end': 35.23221969604492,
   'speaker_id': 2,
   'speaker': 'SPEAKER_02',
   'duration': 3.206249237060547},
  {'start': 35.73846817

In [16]:
result["segments"][:10]

[{'start': 0.9590937495231628,
  'end': 12.737844467163086,
  'speaker_id': 1,
  'speaker': 'SPEAKER_01',
  'duration': 11.778750717639923},
 {'start': 13.328469276428223,
  'end': 20.635345458984375,
  'speaker_id': 1,
  'speaker': 'SPEAKER_01',
  'duration': 7.306876182556152},
 {'start': 22.03597068786621,
  'end': 22.93034553527832,
  'speaker_id': 2,
  'speaker': 'SPEAKER_02',
  'duration': 0.8943748474121094},
 {'start': 23.45347023010254,
  'end': 25.259096145629883,
  'speaker_id': 2,
  'speaker': 'SPEAKER_02',
  'duration': 1.8056259155273438},
 {'start': 26.220970153808594,
  'end': 31.401596069335938,
  'speaker_id': 2,
  'speaker': 'SPEAKER_02',
  'duration': 5.180625915527344},
 {'start': 32.025970458984375,
  'end': 35.23221969604492,
  'speaker_id': 2,
  'speaker': 'SPEAKER_02',
  'duration': 3.206249237060547},
 {'start': 35.738468170166016,
  'end': 40.935970306396484,
  'speaker_id': 2,
  'speaker': 'SPEAKER_02',
  'duration': 5.197502136230469},
 {'start': 41.4590950

In [17]:
print("duration:", result["duration"])
print("num_speakers:", result["num_speakers"])
print("speakers:", result["speakers"])
print("segments:", len(result["segments"]))
print("processing_time:", result["processing_time"])

duration: 118.64534759521484
num_speakers: 2
speakers: ['SPEAKER_01', 'SPEAKER_02']
segments: 26
processing_time: 22.869297900004312
